# acquire

> pull the web, papers, video, files and code into the vault

In [ ]:
#| default_exp acquire

In [ ]:
#| hide
from nbdev.showdoc import *

Acquisition is fossick's job; this module only decides what a document *is* and what provenance to
keep. Every method funnels into `Vault.add`, so identity and de-duplication work the same for a
scraped page as for a local PDF.

In [ ]:
#| export
import re, time
from urllib.parse import urlparse
from fastcore.all import L, Path, patch
from vishalakshi.core import Vault

In [ ]:
#| export
CODE_EXTS = '.py,.js,.ts,.jsx,.tsx,.java,.go,.rs,.rb,.php,.swift,.kt,.scala,.lua,.c,.h,.cpp'

def clip_title(s:str, n:int=120) -> str:
    'Collapse whitespace and clip a scraped title to something a breadcrumb can carry.'
    return re.sub(r'\s+', ' ', (s or '').strip())[:n] or 'untitled'

def _md_title(md:str, fallback:str) -> str:
    "First markdown heading in `md`, else `fallback` — scraped <title>s are often junk."
    m = re.search(r'^#{1,2} +(.+)$', md or '', flags=re.M)
    return clip_title(m.group(1)) if m else clip_title(fallback)

In [ ]:
#| export
@patch
def url(self:Vault,
        url:str,            # page to read
        title:str=None,     # defaults to the page's first heading, else its path
        sel:str=None,       # CSS selector to narrow the page before conversion
        kind:str='web',
        auto:bool=True,     # escalate plain -> heavy -> stealthy -> logged-in Chrome past bot walls
        meta:dict=None,
        force:bool=False,
        **kw                # forwarded to fossick.fetch
) -> dict:
    """Fetch one URL, convert it to markdown and file it in the vault.

    `auto=True` is the default because a bot wall returns HTTP 200 with a challenge page, which
    would otherwise be indexed as if it were the article."""
    from fossick.core import fetch, to_md
    pg = fetch(url, sel=sel, auto=auto, **kw)
    md = to_md(pg, sel=sel)
    if not (md or '').strip(): return dict(url=url, skipped='no readable text', status=getattr(pg, 'status', None))
    m = dict(meta or {}, url=url, status=getattr(pg, 'status', None), tier=getattr(pg, 'tier', None),
             fetched_at=time.time())
    return dict(self.add(md, title or _md_title(md, urlparse(url).path.rsplit('/', 1)[-1] or url),
                         source=url, kind=kind, meta=m, force=force), url=url)

@patch
def crawl(self:Vault, start_url:str, max_pages:int=10, sel:str=None, same_domain:bool=True, **kw) -> list:
    'Crawl a docs site or blog from a start URL and file every page in the vault.'
    from fossick.core import crawl as _crawl, to_md
    out = []
    for pg in _crawl(start_url, sel=sel, max_pages=max_pages, same_domain=same_domain, **kw):
        md = to_md(pg, sel=sel)
        if not (md or '').strip(): continue
        u = pg['url']
        out.append(self.add(md, _md_title(md, u), source=u, kind='web',
                            meta=dict(url=u, via='crawl', root=start_url, fetched_at=time.time())))
    return out

In [ ]:
#| export
@patch
def web(self:Vault,
        query:str,          # what to search for
        n:int=5,            # top results to read
        google:bool=False,  # real Google ranking via a stealth browser (slower)
        sel:str=None,       # CSS selector applied to every result page
        chars:int=60000,    # max markdown chars kept per source
        region:str='us-en',
        **kw                # forwarded to fossick.fetch
) -> dict:
    """Search the web, read the top `n` results, and file all of them in the vault.

    This is the loop the vault exists for: the query that found a page is kept in its metadata, so
    months later `sources()` still says *why* a document is in your corpus. Results already present
    are skipped rather than duplicated, so re-running an overlapping search is cheap."""
    from fossick.search import research
    res = research(query, n=n, engine='google' if google else 'search', sel=sel,
                   chars=chars, region=region, **kw)
    added = []
    for s in res.get('sources', []):
        md, href = s.get('md') or '', s.get('href') or ''
        if not (md.strip() and href): continue
        added.append(dict(self.add(md, clip_title(s.get('title')) or _md_title(md, href),
                                   source=href, kind='web',
                                   meta=dict(url=href, query=query, engine='google' if google else 'ddgs',
                                             fetched_at=time.time())), url=href))
    return dict(query=query, n_found=len(res.get('sources', [])), added=added)

In [ ]:
#| export
@patch
def arxiv(self:Vault, id_or_url:str, save_dir:str=None, force:bool=False, **kw) -> dict:
    'Read an arXiv paper (metadata + full text) into the vault as `kind="arxiv"`.'
    from fossick.core import read_arxiv
    d = save_dir or str(Path(self.path).parent/'pdfs')
    Path(d).mkdir(parents=True, exist_ok=True)
    p = read_arxiv(id_or_url, source=True, save_dir=d, force=force, **kw)
    body = p.get('source') or p.get('summary') or ''
    md = f"# {p['title']}\n\n{p.get('summary','')}\n\n{body}"
    return self.add(md, clip_title(p['title']), source=p.get('link') or id_or_url, kind='arxiv',
                    force=force,
                    meta=dict(authors=list(p.get('authors') or []), published=p.get('published'),
                              pdf_path=p.get('pdf_path'), pdf_url=p.get('pdf_url'),
                              fetched_at=time.time()))

@patch
def pdf(self:Vault, path_or_url:str, title:str=None, force:bool=False, **kw) -> dict:
    'Read a PDF (local path or URL) into the vault, one tree node per heading.'
    from fossick.core import get_pdf
    from litesearch.data import pdf_parse
    p = Path(path_or_url)
    if p.exists(): src, doc = str(p), None
    else:
        doc = get_pdf(path_or_url, **kw)
        if doc is None: return dict(source=path_or_url, skipped='not a PDF or could not be fetched')
        src = path_or_url
    pages = list(enumerate(pdf_parse(doc if doc is not None else src,
                                     out_path=self.assets(Path(src).stem or 'pdf'))))
    ttl = title or clip_title(p.stem.replace('_', ' ') if p.exists() else
                              path_or_url.rsplit('/', 1)[-1].split('?')[0])
    return self.add(pages, ttl, source=src, kind='pdf', force=force,
                    meta=dict(url=None if p.exists() else path_or_url, fetched_at=time.time()))

In [ ]:
#| export
@patch
def youtube(self:Vault, url:str, force:bool=False) -> dict:
    'Read a YouTube video\'s transcript and metadata into the vault.'
    from fossick.core import read_yt
    v = read_yt(url, force=force)
    body = (v.get('source') or '').strip()
    if not body: return dict(source=url, skipped='no transcript available', title=v.get('title'))
    md = f"# {v['title']}\n\n{v.get('description','')}\n\n## Transcript\n\n{body}"
    return self.add(md, clip_title(v['title']), source=url, kind='youtube', force=force,
                    meta=dict(url=url, channel=v.get('channel'), duration=v.get('duration'),
                              upload_date=v.get('upload_date'), fetched_at=time.time()))

In [ ]:
#| export
@patch
def code(self:Vault, dir, types:str=CODE_EXTS, **kw) -> list:
    """File a source tree into the vault as `kind='code'`, so code and prose answer one query.

    This is deliberately the shallow path: files as documents, headings from the text. For call
    graphs, PageRank over symbols and `where_to_add`, point `kosha` at the same repo — it builds an
    AST-derived index that this cannot, and it is a litesearch store too, so both stay queryable."""
    exts = {f".{t.strip().lstrip('.')}".lower() for t in types.split(',')}
    return [self.add_file(p, kind='code', **kw) for p in sorted(Path(dir).rglob('*'))
            if p.is_file() and p.suffix.lower() in exts and not any(
                part.startswith('.') or part in ('node_modules', '__pycache__', 'dist', 'build')
                for part in p.parts)]